# 🏦 Pandas pour auditeurs — Exercices Niveau 3 : Moyen+

**Contexte** : Vous êtes auditeur dans l'équipe **Conformité / LCB-FT** (Lutte contre le Blanchiment
de Capitaux et le Financement du Terrorisme) d'une banque privée.

Le service informatique vous a fourni deux fichiers :
- **`flux`** : les flux financiers de l'exercice (transactions)
- **`clients`** : le référentiel clients avec leur profil de risque KYC

Votre mission : enrichir les données, identifier des comportements atypiques,
et produire un **scoring de risque** par client.

**Compétences couvertes** : `merge`, analyse temporelle (`.dt`), détection de patterns (structuring,
espèces, pays à risque), `groupby` avancé, combinaison de critères, export.

> ℹ️ Les données et les critères d'alerte sont **fictifs et simplifiés** à des fins pédagogiques.

## 0. Génération des données — exécutez en premier

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
np.random.seed(303)

# ── Référentiel clients ───────────────────────────────────────────────────────
n_clients = 60
client_ids = [f'CLI{str(i).zfill(4)}' for i in range(1, n_clients + 1)]

clients = pd.DataFrame({
    'client_id':       client_ids,
    'segment':         np.random.choice(['Particulier', 'Entreprise', 'Private Banking'],
                                        n_clients, p=[0.45, 0.35, 0.20]),
    'pays_residence':  np.random.choice(['FR', 'LU', 'CH', 'MC', 'BE', 'DE'],
                                        n_clients, p=[0.50, 0.15, 0.12, 0.08, 0.10, 0.05]),
    'niveau_risque_kyc': np.random.choice(['Faible', 'Standard', 'Élevé'],
                                          n_clients, p=[0.40, 0.45, 0.15]),
    'date_entree_relation': pd.to_datetime('2015-01-01') + pd.to_timedelta(
        np.random.randint(0, 3285, n_clients), unit='D'
    ),
})

# ── Transactions ──────────────────────────────────────────────────────────────
n = 600

pays_contrepartie = np.random.choice(
    ['FR', 'DE', 'BE', 'LU', 'CH', 'US', 'GB', 'AE', 'PA', 'CY', 'MT', 'SG'],
    n, p=[0.20, 0.12, 0.10, 0.10, 0.08, 0.08, 0.07, 0.06, 0.05, 0.05, 0.05, 0.04]
)

types_op = np.random.choice(
    ['Virement entrant', 'Virement sortant', 'Espèces dépôt', 'Espèces retrait',
     'Chèque', 'Prélèvement'],
    n, p=[0.28, 0.28, 0.10, 0.10, 0.12, 0.12]
)

dates = pd.to_datetime('2024-01-01') + pd.to_timedelta(
    np.random.randint(0, 366, n), unit='D'
)

montants_base = np.round(np.random.lognormal(mean=7.0, sigma=1.3, size=n), 2)

flux = pd.DataFrame({
    'flux_id':          [f'FL{str(i).zfill(6)}' for i in range(1, n + 1)],
    'client_id':        np.random.choice(client_ids, n),
    'date':             dates,
    'type_operation':   types_op,
    'montant':          montants_base,
    'devise':           np.random.choice(['EUR', 'USD', 'CHF', 'GBP'],
                                         n, p=[0.78, 0.10, 0.08, 0.04]),
    'pays_contrepartie': pays_contrepartie,
    'canal':            np.random.choice(['SWIFT', 'SEPA', 'Interne', 'Guichet'],
                                         n, p=[0.25, 0.40, 0.20, 0.15]),
})

# ── Pièges volontaires ────────────────────────────────────────────────────────
# Structuring : montants juste sous 10 000
idx_struct = np.random.choice(flux.index, 12, replace=False)
flux.loc[idx_struct, 'montant'] = np.random.choice([9500, 9750, 9800, 9900, 9950, 9990], 12)
flux.loc[idx_struct, 'type_operation'] = 'Espèces dépôt'

# Espèces importantes
idx_cash = np.random.choice(flux.index, 8, replace=False)
flux.loc[idx_cash, 'montant'] = np.random.choice([15000, 20000, 25000, 30000], 8)
flux.loc[idx_cash, 'type_operation'] = np.random.choice(['Espèces dépôt', 'Espèces retrait'], 8)

# Pays à risque (liste fictive simplifiée)
pays_risque = ['AE', 'PA', 'CY']
idx_risque = np.random.choice(flux.index, 15, replace=False)
flux.loc[idx_risque, 'pays_contrepartie'] = np.random.choice(pays_risque, 15)
flux.loc[idx_risque, 'montant'] = np.round(np.random.lognormal(mean=9.0, sigma=0.8, size=15), 2)

# Valeurs manquantes
flux.loc[np.random.choice(flux.index, 15, replace=False), 'pays_contrepartie'] = np.nan

flux = flux.sample(frac=1, random_state=5).reset_index(drop=True)

print('Flux prêt :', flux.shape)
print('Clients prêt :', clients.shape)

**Description des tables**

**Table `flux`**

| Colonne | Description |
|---|---|
| `flux_id` | identifiant du flux |
| `client_id` | identifiant client |
| `date` | date du flux |
| `type_operation` | Virement entrant/sortant, Espèces dépôt/retrait, Chèque, Prélèvement |
| `montant` | montant en devise |
| `devise` | EUR / USD / CHF / GBP |
| `pays_contrepartie` | code pays ISO de la contrepartie |
| `canal` | SWIFT / SEPA / Interne / Guichet |

**Table `clients`**

| Colonne | Description |
|---|---|
| `client_id` | identifiant client |
| `segment` | Particulier / Entreprise / Private Banking |
| `pays_residence` | pays de résidence |
| `niveau_risque_kyc` | Faible / Standard / Élevé |
| `date_entree_relation` | date d'entrée en relation |

**Pays à risque (liste fictive simplifiée)** : `AE`, `PA`, `CY`

---
## Exercice 1 — Enrichissement par `merge`

Avant d'analyser les flux, il faut **ramener les informations clients** dans la table des flux.

**Questions :**
1. Fusionnez `flux` et `clients` sur `client_id` (jointure gauche — `how='left'`).
   Appelez le résultat `flux_enrichi`. Vérifiez le nombre de colonnes obtenu.
2. Y a-t-il des flux pour lesquels le client est **introuvable** dans le référentiel ?
   (vérifiez les `NaN` dans la colonne `niveau_risque_kyc` après le merge)
3. Quel est le **montant total** des flux par **niveau de risque KYC** ?
   Triez du plus élevé au plus faible.
4. Combien de flux concernent des clients de **`Private Banking`** résidant **hors de France** ?

In [ ]:
# 1. Merge flux + clients
# Votre code ici


In [ ]:
# 2. Flux avec client introuvable
# Votre code ici


In [ ]:
# 3. Montant total par niveau de risque KYC
# Votre code ici


In [ ]:
# 4. Flux Private Banking hors France
# Votre code ici


---
## Exercice 2 — Analyse temporelle

En LCB-FT, le **moment** des transactions peut être un signal d'alerte.

**Questions :**
1. Extrayez dans `flux_enrichi` les colonnes **`mois`**, **`jour_semaine`** (0=lundi … 6=dimanche)
   et **`nom_jour`** depuis la colonne `date`.
2. Calculez le **montant total par mois**. Quel mois est le plus actif en volume ?
3. Isolez les flux réalisés le **week-end** (samedi ou dimanche). Combien représentent-ils
   en nombre et en pourcentage du total ?
4. Parmi les flux du week-end, lesquels sont des **`Espèces dépôt`** ou **`Espèces retrait`** ?
   Affichez les colonnes `date`, `nom_jour`, `client_id`, `type_operation`, `montant`.

In [ ]:
# 1. Extraire mois, jour_semaine, nom_jour
# Votre code ici


In [ ]:
# 2. Montant total par mois
# Votre code ici


In [ ]:
# 3. Flux du week-end (nombre et %)
# Votre code ici


In [ ]:
# 4. Espèces le week-end
# Votre code ici


---
## Exercice 3 — Détection du *structuring* (fractionnement)

Le **structuring** consiste à fractionner des flux en montants juste inférieurs
au seuil de déclaration de **10 000 €** pour éviter les contrôles.

**Questions :**
1. Identifiez les flux dont le montant est **entre 9 000 et 9 999,99 €** (inclus).
   Combien y en a-t-il ?
2. Parmi ces flux suspects, quel est le type d'opération le plus fréquent ?
3. Pour chaque **client** ayant au moins **2 flux** dans la zone 9 000–9 999,
   calculez le nombre de ces flux et le montant total.
   Ce client mérite une attention particulière — quel est son `niveau_risque_kyc` ?
4. Créez une colonne booléenne **`alerte_structuring`** dans `flux_enrichi` :
   `True` si le montant est entre 9 000 et 9 999 **ET** l'opération est une Espèce.

In [ ]:
# 1. Flux en zone 9 000 – 9 999
# Votre code ici


In [ ]:
# 2. Type d'opération le plus fréquent parmi les suspects
# Votre code ici


In [ ]:
# 3. Clients avec >= 2 flux en zone 9 000–9 999
# Votre code ici


In [ ]:
# 4. Colonne alerte_structuring
# Votre code ici


---
## Exercice 4 — Analyse des flux en espèces

Les opérations en espèces sont particulièrement surveillées en LCB-FT.

**Questions :**
1. Isolez tous les flux de type **`Espèces dépôt`** ou **`Espèces retrait`**.
2. Pour chaque **client**, calculez le **cumul** des montants en espèces (toutes opérations confondues).
   Identifiez les 10 clients avec le cumul le plus élevé. Quel est leur `niveau_risque_kyc` ?
3. Créez une colonne **`alerte_especes`** dans `flux_enrichi` :
   `True` si le flux est une espèce **ET** le montant dépasse **10 000 €**.
4. Combien de clients distincts ont au moins **une** alerte espèces ?

In [ ]:
# 1. Flux en espèces
# Votre code ici


In [ ]:
# 2. Top 10 clients par cumul espèces
# Votre code ici


In [ ]:
# 3. Colonne alerte_especes
# Votre code ici


In [ ]:
# 4. Nombre de clients avec alerte espèces
# Votre code ici


---
## Exercice 5 — Flux vers pays à risque

La liste fictive des **pays à risque** pour cet exercice est : `['AE', 'PA', 'CY']`.

**Questions :**
1. Créez une colonne **`pays_risque`** dans `flux_enrichi` : `True` si `pays_contrepartie`
   est dans la liste des pays à risque.
2. Quel est le **montant total** des flux vers des pays à risque, par **type d'opération** ?
3. Identifiez les clients ayant **au moins 3 flux** vers des pays à risque.
   Affichez leur `client_id`, `segment`, `niveau_risque_kyc`, nombre de flux et montant total.
4. Parmi les flux vers pays à risque, quels **clients à risque KYC `Élevé`** y participent ?

In [ ]:
# 1. Colonne pays_risque
pays_risque_liste = ['AE', 'PA', 'CY']
# Votre code ici


In [ ]:
# 2. Montant total vers pays à risque par type d'opération
# Votre code ici


In [ ]:
# 3. Clients avec >= 3 flux vers pays à risque
# Votre code ici


In [ ]:
# 4. Clients KYC Élevé impliqués dans des flux vers pays à risque
# Votre code ici


---
## Exercice 6 — Scoring de risque multi-critères (synthèse)

Vous devez produire un **tableau de bord de risque** par client, en combinant toutes les alertes.

**Construisez une table avec, pour chaque client :**
- `nb_flux_total` : nombre total de flux
- `montant_total` : montant total
- `nb_alertes_structuring` : nombre de flux avec `alerte_structuring == True`
- `nb_alertes_especes` : nombre de flux avec `alerte_especes == True`
- `nb_flux_pays_risque` : nombre de flux vers pays à risque
- `score_risque` : somme des trois compteurs d'alertes
- `niveau_risque_kyc` : ramené depuis la table `clients`
- `segment` : ramené depuis la table `clients`

Triez par `score_risque` décroissant et affichez les **10 clients les plus à risque**.

Exportez ce tableau dans un fichier **`rapport_risque_lcbft.xlsx`**.

In [ ]:
# Scoring de risque multi-critères
# Votre code ici


In [ ]:
# Export Excel
# Votre code ici
